In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import json

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])



sk-proj-


### Setup PushOver (send phone notification)

In [ ]:
# Step 1 - setup an account in PushOver
# Step 2- Setup the app on our iphone / android phone, log into the same account
# step 3 - create an "Application/Api Token" from the browser
# step 4 - copy your user key and api token into the .env file and save the changes
# e.g PUSHOVER_USER= xxxxxxxxx PUSHOVER_TOKEN=yyyyyyyyy
# PUSHOVER_USER=uj2zzdtmw5jhsyhrenptcbut58wgzs
# PUSHOVER_TOKEN=a17pqtdb46zrxux3tughsdkykbqcgd

load_dotenv()
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print(pushover_user)
print(pushover_token)

In [4]:
import requests

def send_notification(message:str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [11]:
send_notification ("Hello, I'm in a class of AI Engineering training")

### Use pushover using LLM tool

In [17]:
send_notification_function = {

    "name" : "send_notification",
    "description": "Sends a push notification ot the user's phone via pushover. Use this to alert the user about the change",
    "parameters": {
        "type": "object",
        "properties": {

            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required":["message"]
    }
}

### Add Pushover to the list of tools for the LLM (so far only one tool)

In [18]:
tools = [{"type": "function", "function": send_notification_function}]

### CALLING THE TOOL FROM AN LLM

In [19]:
client = OpenAI()
response = client.chat.completions.create(

    model = "gpt-4.1-mini",
    messages = [
        { "role":"user", "content": "Send me a notificaiton about the AI engineering course"}
    ],
    tools = tools
)

#check if model wants to call a tool
message = response.choices[0].message
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_OC9lFvlA2g1DkiydYN2F3YL6', function=Function(arguments='{"message":"Reminder: Don\'t forget to check out the AI Engineering course for the latest updates and materials!"}', name='send_notification'), type='function')])


In [ ]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    #send notification
    send_notification(args["message"])
    print(f"Sent notification: {args["message"]}")
else:
    print(message.content)

Sent notification: Reminder: Don't forget to check out the AI Engineering course for the latest updates and materials!
